# d vs b check

**Question:** for covered targets, direct transfer uses `d = y_K562(t) − mean K562 response`. If a target barely does anything in K562, `y ≈ 0`, so `d ≈ −mean`, which points against `b`. That would explain why `d` came out *below* chance on HCT116's real significant genes (0.42 vs `b` at 0.65).

**Supported if** the weakest strength bin (Q1) has `corr_d_b` clearly negative, `cos_d_negm` near 1, and the lowest `real_acc_d`, with the stronger bins better.

Part 1 tests it on HCT116 (real cells). Part 2 asks how many **test-panel** targets are in the weak regime, which says how much a fix could matter. Same model settings as the submitted +0.0265 run.

In [1]:
import warnings
import numpy as np
import pandas as pd
import anndata as ad

from model import TransferModel
from config import SlimConfig
from data_processing import DataProcessor
from real_score import load_line, set_line_context, sign_diagnostic, d_vs_b_check

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)

DATASETS = ["K562", "RPE1", "HepG2", "Jurkat"]
K, LAM, ALPHA, GAIN, TRANSFER_C = 20, 100, 2.0, 2.5, 200   # as in the +0.0265 submission

In [2]:
gene_names = pd.read_csv("../../data/controls/gene_names.csv")["gene_name"].tolist()
dp = DataProcessor(gene_names)

model = TransferModel(gene_names, "../../data/embeddings/gene_string_embeddings.v0.3.h5",
                      SlimConfig(), transfer_c=TRANSFER_C)
model.config.K, model.config.lambda_reg = K, LAM
model.fit(*dp.build_Y(DATASETS, genes="union"), {n: dp.load_dataset(n).var_names for n in DATASETS})

  Fitted on 15627 perts, STRING coverage: 15118/15627
  direct transfer from ('K562',): 9675 targets have a measurement


## Part 1: HCT116 (real cells, all 198 targets covered by K562)

In [3]:
hct = load_line("HCT116", gene_names)
set_line_context(model, hct, dp, DATASETS)
diag, diag_summary = sign_diagnostic(model, hct, {"gain": GAIN}, kd=0.9, alpha=ALPHA)   # ~3 min
diag_summary.round(3)

HCT116: 18400 controls, 198 targets (cells per target median 184, min 100), 100 dropped (7s)
HCT116 {'K562': np.float64(0.27), 'RPE1': np.float64(0.3), 'HepG2': np.float64(0.19), 'Jurkat': np.float64(0.24)}
sign_diagnostic: 198 targets (182s)


n_calls            384.000
n_real               2.000
acc_model            0.522
acc_E                0.521
realized             0.979
acc_b                0.522
acc_d                0.495
acc_shift            0.513
b_vs_shift           0.591
d_vs_resid           0.512
b_dom                0.890
acc_bdom             0.525
acc_ddom             0.502
lib_ratio            0.969
real_acc_E           0.649
real_acc_b           0.653
real_acc_d           0.422
real_acc_shift       0.615
real_d_vs_resid      0.422
dtype: float64

In [4]:
hct_df, hct_bins = d_vs_b_check(model, hct.targets, diag=diag)
hct_bins.round(3)

,strength,n_targets,lam,corr_d_b,cos_d_negm,acc_d,real_acc_d,real_acc_b
strength_bin,,,,,,,,
Q1,5.684,45,0.586,-0.106,0.157,0.499,0.435,0.692
Q2,6.825,44,0.506,-0.062,0.098,0.485,0.384,0.694
Q3,7.924,44,0.435,-0.054,0.071,0.499,0.447,0.658
Q4,10.448,45,0.315,-0.004,-0.016,0.503,0.418,0.644


## Part 2: test panel (context D controls, no real cells)

In [5]:
test_targets = pd.read_csv("../../data/vcc_2026_controls_final/pert_counts.csv")["target_gene"].tolist()
ctx = ad.read_h5ad("../../data/vcc_2026_controls_final/context_D.h5ad", backed="r")
profile = pd.Series(dp.context_ctrl_mean(ctx), index=gene_names)
model.set_context(profile, dp.source_ctrls(DATASETS), tau=0.05, mix=0.3, panel=test_targets)

test_df, test_bins = d_vs_b_check(model, test_targets)
q1_edge = hct_df.groupby("strength_bin", observed=True)["strength"].max().iloc[0]
print(f"{len(test_df)} of {len(test_targets)} test targets covered by K562")
print(f"share of test targets at or below HCT116's Q1 strength ({q1_edge:.2f}):",
      round(float((test_df["strength"] <= q1_edge).mean()), 3))
test_bins.round(3)

272 of 300 test targets covered by K562
share of test targets at or below HCT116's Q1 strength (6.31): 0.228


,strength,n_targets,lam,corr_d_b,cos_d_negm
strength_bin,,,,,
Q1,5.725,68,0.583,-0.116,0.170
Q2,6.964,68,0.498,-0.068,0.107
Q3,8.350,68,0.439,-0.032,0.038
Q4,10.672,68,0.304,0.001,-0.011
